# 📏 Model Evaluation & Prediction Accuracy
**Foundations · Data Pattern Recognition for the Rest of Us**

> Building a model is only half the job. Knowing *how good* it is — and whether it will hold up on new data — is the other half. This notebook gives you the complete toolkit for evaluating any model you build in this course.

### What you'll learn
- The cardinal rule: always evaluate on **both** Training and Test sets
- Regression metrics: MSE, RMSE, MAE, MAPE — when to use which
- Classification metrics: Accuracy, Recall, Precision, F1, AUC — and why Accuracy alone is dangerous
- How to detect and diagnose **overfitting**
- A systematic framework for comparing multiple models

### Time: ~45 minutes

In [ ]:
# ⚠️  Run this cell first
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from scipy import stats

print(f"✓ Setup complete — Python {sys.version.split()[0]}")

## 📐 Part 1 — The Cardinal Rule: Train vs Test

Every evaluation in this course follows one non-negotiable rule:

> **Never evaluate a model on the same data it was trained on.**

A model can memorise its training data and score 100% — this is called **overfitting**. It means the model has learned noise, not signal, and will perform poorly on any new data you give it.

The solution: split your data before training. Train on one portion, evaluate on the other.

```
Full dataset
├── Training set (70–80%) ← model learns from this
└── Test set    (20–30%) ← model is evaluated on this
                            (model has NEVER seen this data)
```

The gap between training and test performance is your overfitting detector.

In [ ]:
# Demonstrate the train/test split and why it matters
# Synthetic regression dataset: true relationship y = 2x + noise
np.random.seed(42)
n = 150
X = np.sort(np.random.uniform(0, 10, n))
y = 2 * X + np.random.normal(0, 2, n)

X_tr, X_te, y_tr, y_te = train_test_split(
    X.reshape(-1,1), y, test_size=0.25, random_state=42)

# Fit three models of increasing complexity
models = {
    'Linear (degree 1)':    1,
    'Polynomial (degree 5)': 5,
    'Overfit (degree 15)':  15,
}

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

results = []
x_plot = np.linspace(0, 10, 300).reshape(-1,1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#1e5fa8', '#1a7a45', '#e85d20']

for ax, (name, degree), color in zip(axes, models.items(), colors):
    pipe = Pipeline([('poly', PolynomialFeatures(degree)),
                     ('lr',   LinearRegression())])
    pipe.fit(X_tr, y_tr)

    train_rmse = np.sqrt(mean_squared_error(y_tr, pipe.predict(X_tr)))
    test_rmse  = np.sqrt(mean_squared_error(y_te, pipe.predict(X_te)))
    gap        = test_rmse - train_rmse

    results.append({'Model': name, 'Train RMSE': train_rmse,
                    'Test RMSE': test_rmse, 'Gap': gap})

    ax.scatter(X_tr, y_tr, color=color, alpha=0.4, s=18, label='Train')
    ax.scatter(X_te, y_te, color='#888', alpha=0.5, s=18, label='Test')
    ax.plot(x_plot, pipe.predict(x_plot), color=color, lw=2.5)
    ax.set_xlim(0, 10); ax.set_ylim(-5, 25)
    ax.set_title(f'{name}\nTrain RMSE={train_rmse:.2f}  Test RMSE={test_rmse:.2f}',
                 fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('The Same Data — Three Very Different Models\n'
             'Low complexity underfits. High complexity overfits. The middle generalises.',
             fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

df_results = pd.DataFrame(results)
print(df_results.round(3).to_string(index=False))
print("\n📌 The degree-15 polynomial has near-zero training error but terrible test error.")
print("   It memorised noise. The linear model generalises far better.")

## 📐 Part 2 — Regression Metrics

When your target is a **continuous number** (price, salary, sales), use these metrics.
All four measure the gap between predicted and actual values — **lower is better**.

| Metric | Formula | Key property |
|--------|---------|-------------|
| **MSE** | mean((ŷ - y)²) | Penalises large errors heavily (squared) |
| **RMSE** | √MSE | Same units as target — most interpretable |
| **MAE** | mean(|ŷ - y|) | Robust to outliers; average absolute error |
| **MAPE** | mean(|ŷ - y| / y) × 100 | Scale-free %; avoid when target has zeros |

**When to use which:**
- Report **RMSE** as your headline metric — it's in the same units as what you're predicting
- Use **MAE** when outliers are common and you don't want them to dominate
- Use **MAPE** when you want to communicate error as a percentage to stakeholders
- **MSE** is mostly used internally (optimisation) — hard to interpret because it's squared

In [ ]:
# Regression metrics — visual and numerical comparison
np.random.seed(7)
n = 100
y_true  = 50 + 10 * np.random.randn(n)           # true values ~N(50, 10)
y_pred  = y_true + np.random.normal(0, 5, n)      # predictions with moderate error

# Add a few large outliers
y_pred_outliers = y_pred.copy()
y_pred_outliers[[3, 25, 67]] += [40, -35, 30]     # three big misses

def regression_metrics(y_true, y_pred, label):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    print(f"  {label}:")
    print(f"    MSE  = {mse:.2f}   RMSE = {rmse:.2f}   MAE = {mae:.2f}   MAPE = {mape:.1f}%   R² = {r2:.3f}")
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R²': r2}

print("=== Regression Metrics Comparison ===\n")
m1 = regression_metrics(y_true, y_pred, "Model A — moderate error, no outliers")
m2 = regression_metrics(y_true, y_pred_outliers, "Model B — same error + 3 large outliers")

print("\n📌 RMSE jumps dramatically with outliers (squared error amplifies them)")
print("   MAE barely changes — it treats all errors equally")
print("   → If outliers matter, report RMSE.  If they don't, report MAE.")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, pred, title in [(axes[0], y_pred, 'Model A — No outliers'),
                         (axes[1], y_pred_outliers, 'Model B — With outliers')]:
    residuals = y_true - pred
    ax.scatter(pred, residuals, alpha=0.6, color='#1e5fa8', s=20)
    ax.axhline(0, color='#e85d20', lw=1.5, ls='--')
    ax.set_xlabel('Predicted value'); ax.set_ylabel('Residual (actual − predicted)')
    ax.set_title(f'Residual Plot — {title}')

plt.tight_layout(); plt.show()
print("\n📌 Good model: residuals scattered randomly around zero, no pattern")
print("   Pattern in residuals = the model is missing something systematic")

## 📐 Part 3 — Classification Metrics

When your target is a **category** (yes/no, fraud/not-fraud, churn/stay), use these.

### The Confusion Matrix

Every classification prediction falls into one of four buckets:

|  | Predicted Positive | Predicted Negative |
|--|---|---|
| **Actually Positive** | True Positive (TP) ✓ | False Negative (FN) ✗ |
| **Actually Negative** | False Positive (FP) ✗ | True Negative (TN) ✓ |

From these four numbers, all classification metrics are derived:

| Metric | Formula | When it matters |
|--------|---------|----------------|
| **Accuracy** | (TP+TN)/(all) | Only when classes are balanced |
| **Recall/Sensitivity** | TP/(TP+FN) | When missing a positive is costly (fraud, cancer) |
| **Precision** | TP/(TP+FP) | When false alarms are costly (spam filter) |
| **Specificity** | TN/(TN+FP) | When true negatives matter (medical screening) |
| **F1 Score** | 2·(P·R)/(P+R) | Best single metric for **imbalanced** classes |
| **AUC** | — | Works across all thresholds; 0.5=random, 1.0=perfect |

### ⚠️ The Accuracy Trap
If 95% of transactions are legitimate and 5% are fraud, a model that always predicts "not fraud" achieves **95% accuracy** — while catching **zero fraud**. Always check F1 and AUC for imbalanced problems.

In [ ]:
# The Accuracy Trap — demonstrated with imbalanced data
np.random.seed(42)

# Imbalanced dataset: 95% negative, 5% positive (like fraud)
n = 1000
y_true_imb = np.array([0]*950 + [1]*50)

# "Dumb" model: always predicts 0 (no fraud)
y_pred_dumb = np.zeros(n, dtype=int)

# "Smart" model: actually learns something
noise = np.random.normal(0, 1, n)
score = np.where(y_true_imb == 1,
                 np.random.uniform(0.4, 0.9, n),
                 np.random.uniform(0.05, 0.5, n))
y_pred_smart = (score > 0.5).astype(int)
y_score_smart = score

print("=== The Accuracy Trap ===\n")
print(f"{'Metric':<20} {'Always-0 model':>18} {'Smart model':>14}")
print("-" * 55)
for metric, func, kwargs in [
    ('Accuracy',    accuracy_score,  {}),
    ('Recall',      recall_score,    {'zero_division':0}),
    ('Precision',   precision_score, {'zero_division':0}),
    ('F1 Score',    f1_score,        {'zero_division':0}),
]:
    d = func(y_true_imb, y_pred_dumb, **kwargs)
    s = func(y_true_imb, y_pred_smart, **kwargs)
    print(f"  {metric:<18} {d:>18.3f} {s:>14.3f}")

auc_smart = roc_auc_score(y_true_imb, y_score_smart)
print(f"  {'AUC':<18} {'N/A':>18} {auc_smart:>14.3f}")

print("\n📌 The dumb model has 95% accuracy — but catches ZERO fraud (Recall=0)")
print("   The smart model has lower accuracy but catches actual fraud (Recall>0)")
print("   For imbalanced problems: always report F1 and AUC, not just Accuracy")

# Visualise confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pred, title in [(axes[0], y_pred_dumb,  'Always-0 Model (95% accuracy, 0% recall)'),
                         (axes[1], y_pred_smart, f'Smart Model (AUC={auc_smart:.2f})')]:
    cm = confusion_matrix(y_true_imb, pred)
    ConfusionMatrixDisplay(cm, display_labels=['Not Fraud','Fraud']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ROC Curve and AUC — understanding what AUC means
np.random.seed(42)
n = 500

# Three models with different discrimination ability
y_true_roc = np.array([0]*375 + [1]*125)
scores = {
    'Random guess (AUC=0.5)':    np.random.uniform(0, 1, n),
    'Decent model (AUC≈0.75)':   np.where(y_true_roc==1,
                                   np.random.beta(3,2,n),
                                   np.random.beta(2,3,n)),
    'Strong model (AUC≈0.92)':   np.where(y_true_roc==1,
                                   np.random.beta(8,2,n),
                                   np.random.beta(2,8,n)),
}

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#888', '#e85d20', '#1a7a45']
for (name, score), color in zip(scores.items(), colors):
    auc = roc_auc_score(y_true_roc, score)
    RocCurveDisplay.from_predictions(y_true_roc, score,
        name=f'{name.split("(")[0].strip()} (AUC={auc:.2f})',
        ax=ax, color=color)

ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4,label='Random (AUC=0.50)')
ax.set_title('ROC Curves — Three Models\n'
             'Closer to top-left corner = better discrimination')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print("\n📌 AUC interpretation:")
print("   0.50 = no better than random guessing")
print("   0.70 = acceptable for many business problems")
print("   0.80 = good")
print("   0.90 = excellent")
print("   1.00 = perfect (usually means data leakage — be suspicious!)")

## 📐 Part 4 — The Model Comparison Framework

When comparing multiple models, follow this systematic approach:

**Step 1 — Use the SAME split for all models**
Never re-split between models. Different splits give incomparable results.

**Step 2 — Record both Train AND Test scores**
A model with great test performance but huge train-test gap is overfit and fragile.

**Step 3 — Choose metrics appropriate for your problem**
- Regression → RMSE as headline, MAE as secondary
- Classification balanced → Accuracy + AUC
- Classification imbalanced → F1 + AUC, Accuracy is misleading

**Step 4 — Consider more than just accuracy**
- Interpretability: can you explain the prediction?
- Speed: how fast does it predict at inference time?
- Stability: does performance hold across different random seeds?

**Step 5 — Diagnose the gap**
- Gap < 0.01 → good generalisation
- Gap 0.01–0.05 → monitor, consider regularisation
- Gap > 0.05 → overfitting — reduce complexity or add regularisation

In [ ]:
# Full model comparison — regression example
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

np.random.seed(42)
n = 500
X_data = np.random.randn(n, 8)
y_data = (3*X_data[:,0] - 2*X_data[:,1] + X_data[:,2]**2
          + 0.5*X_data[:,3]*X_data[:,4] + np.random.normal(0, 1.5, n))

X_tr, X_te, y_tr, y_te = train_test_split(X_data, y_data,
                                            test_size=0.25, random_state=42)

models_to_compare = {
    'Linear Regression':          LinearRegression(),
    'Ridge (α=1)':                Ridge(alpha=1.0),
    'Decision Tree (depth=3)':    DecisionTreeRegressor(max_depth=3, random_state=42),
    'Decision Tree (depth=15)':   DecisionTreeRegressor(max_depth=15, random_state=42),
    'Random Forest':              RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':          GradientBoostingRegressor(n_estimators=100, random_state=42),
}

rows = []
for name, model in models_to_compare.items():
    model.fit(X_tr, y_tr)
    tr_rmse = np.sqrt(mean_squared_error(y_tr, model.predict(X_tr)))
    te_rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te)))
    gap     = te_rmse - tr_rmse
    verdict = ('✓ OK' if abs(gap) < 0.1
               else ('~ Monitor' if abs(gap) < 0.5 else '⚠ Overfit'))
    rows.append({'Model': name, 'Train RMSE': round(tr_rmse,3),
                 'Test RMSE': round(te_rmse,3),
                 'Gap': round(gap,3), 'Verdict': verdict})

df_cmp = pd.DataFrame(rows).sort_values('Test RMSE')
print("=== Regression Model Comparison ===")
print(df_cmp.to_string(index=False))

# Visualise
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(df_cmp))
w = 0.35
bars1 = ax.bar(x - w/2, df_cmp['Train RMSE'], w, label='Train RMSE',
               color='#1e5fa8', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + w/2, df_cmp['Test RMSE'],  w, label='Test RMSE',
               color='#e85d20', alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(df_cmp['Model'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('RMSE')
ax.set_title('Model Comparison — Train vs Test RMSE\n'
             'Large gap between bars = overfitting')
ax.legend()
plt.tight_layout(); plt.show()

best = df_cmp.iloc[0]
print(f"\n📌 Best test RMSE: {best['Model']} ({best['Test RMSE']:.3f})")
print(f"   Decision Tree depth=15 has the lowest TRAIN RMSE but worst TEST RMSE — classic overfit")

## 📊 Accuracy Reference Card

Quick reference for every model evaluation you'll do in this course.

| Problem type | Primary metric | Secondary | When Accuracy is misleading |
|---|---|---|---|
| Regression | RMSE | MAE | N/A |
| Classification (balanced) | Accuracy + AUC | F1 | Rarely |
| Classification (imbalanced) | F1 + AUC | Recall/Precision | **Always** |
| Fraud / Medical screening | Recall | AUC | **Always** |
| Spam / False alarm sensitive | Precision | F1 | **Always** |

**Overfitting quick check:**
```
gap = Test RMSE − Train RMSE   (regression)
gap = Train Acc − Test Acc     (classification)

gap < 0.01  →  ✓ Good
gap 0.01–0.05  →  ~ Monitor  
gap > 0.05  →  ⚠ Overfit
```

## 💼 Exercise

Using the comparison table built in Part 4:

**Task 1:** Add a classification comparison. Generate a binary dataset with `make_classification(n_samples=500, random_state=42)` and compare Logistic Regression, Decision Tree (depth=3), Decision Tree (depth=15), and Random Forest using Accuracy, F1, and AUC.

**Task 2:** Make the dataset imbalanced — use `weights=[0.95, 0.05]` in `make_classification`. Re-run the comparison. Which metric changes the most? Which model is most affected?

In [ ]:
# YOUR CODE HERE
from sklearn.datasets import make_classification

# Task 1: balanced classification comparison
# X, y = make_classification(n_samples=500, n_features=10, random_state=42)
# ...

# Task 2: imbalanced version
# X_imb, y_imb = make_classification(n_samples=500, n_features=10,
#                                     weights=[0.95, 0.05], random_state=42)
# ...
print("Replace this with your code")

In [ ]:
# @title 📝 Quiz — Model Evaluation & Prediction Accuracy
# @markdown Answer each question in the box below, then run the AI grading cell.
# @markdown
# @markdown **Q1:** A model scores 98% training accuracy and 62% test accuracy. What does this tell you?
# @markdown **Q2:** You are building a fraud detection model. Only 0.5% of transactions are fraud. Why is Accuracy a poor metric here?
# @markdown **Q3:** What is the difference between Precision and Recall? Give a business example where you would prioritise each.
# @markdown **Q4:** Your regression model has RMSE=5.2 on training data and RMSE=5.4 on test data. Is this a concern?
# @markdown **Q5:** What does an AUC of 0.50 mean? What about 0.95?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_="Model Evaluation & Prediction Accuracy"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question say CORRECT, PARTIAL, or INCORRECT and one sentence why.")
    print("Accept any reasonable paraphrase as correct.")
    print("End with an overall grade (Excellent / Good / Needs Review / Incomplete)")
    print("and one specific concept I should review if I struggled.")
    print()
    print("After grading, I will paste outputs and charts from the notebook —")
    print("please help me understand them and answer any follow-up questions.")

---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong. This is how real open-source projects work.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*